# 04 — comparative analysis

Este notebook compara v1, v2 y v3 con el mismo protocolo de evaluación.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.append(str(PROJECT_ROOT))

import json
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from src.utils.io import load_json
from src.evaluation.reporting import plot_confusion

In [ ]:
MODEL_DIRS = {
    "v1": PROJECT_ROOT / "models" / "v1",
    "v2": PROJECT_ROOT / "models" / "v2",
    "v3": PROJECT_ROOT / "models" / "v3",
}

available = {k: d for k, d in MODEL_DIRS.items() if (d / "metrics_test.json").exists() and (d / "metadata.json").exists()}
available

## Carga de métricas

Si todavía no has ejecutado v1/v2/v3, este notebook servirá como plantilla y no mostrará resultados finales.

In [ ]:
rows = []
loaded = {}

for version, model_dir in available.items():
    metrics = load_json(model_dir / "metrics_test.json")
    metadata = load_json(model_dir / "metadata.json")
    model = tf.keras.models.load_model(model_dir / "model.keras", compile=False)

    row = {
        "version": version,
        "accuracy": metrics["global_metrics"]["accuracy"],
        "macro_f1": metrics["global_metrics"]["macro_f1"],
        "weighted_f1": metrics["global_metrics"]["weighted_f1"],
        "macro_precision": metrics["global_metrics"]["macro_precision"],
        "macro_recall": metrics["global_metrics"]["macro_recall"],
        "macro_ovr_roc_auc": metrics["global_metrics"].get("macro_ovr_roc_auc"),
        "parametros": int(model.count_params()),
        "input_shapes": metadata["input_shapes"],
    }

    inf = metrics.get("inference_time", {})
    row["inference_mean_s"] = inf.get("mean_seconds")
    row["inference_std_s"] = inf.get("std_seconds")

    loaded[version] = {"metrics": metrics, "metadata": metadata, "model": model}
    rows.append(row)

comparison_df = pd.DataFrame(rows).sort_values("macro_f1", ascending=False) if rows else pd.DataFrame()
comparison_df

## Tabla comparativa global

In [ ]:
comparison_df

## Métricas por clase

In [ ]:
if loaded:
    per_class_rows = []
    for version, pack in loaded.items():
        report = pack["metrics"]["classification_report"]
        for class_name, metrics_dict in report.items():
            if class_name in ("accuracy", "macro avg", "weighted avg"):
                continue
            per_class_rows.append({
                "version": version,
                "class_name": class_name,
                "precision": metrics_dict["precision"],
                "recall": metrics_dict["recall"],
                "f1_score": metrics_dict["f1-score"],
                "support": metrics_dict["support"],
            })
    per_class_df = pd.DataFrame(per_class_rows)
    per_class_df
else:
    pd.DataFrame()

## Curvas y AUC por clase

In [ ]:
if loaded:
    curve_rows = []
    for version, pack in loaded.items():
        curves = pack["metrics"]["curves"]
        class_names = {int(k): v for k, v in pack["metadata"]["class_names"].items()}
        for cls, name in class_names.items():
            curve_rows.append({
                "version": version,
                "class_name": name,
                "roc_auc": curves["roc_auc"].get(str(cls), curves["roc_auc"].get(cls)),
                "pr_auc": curves["pr_auc"].get(str(cls), curves["pr_auc"].get(cls)),
            })
    pd.DataFrame(curve_rows)
else:
    pd.DataFrame()

## Coste de inferencia y número de parámetros

In [ ]:
comparison_df[["version", "parametros", "inference_mean_s", "inference_std_s"]] if not comparison_df.empty else comparison_df

## Robustez por registros

Este bloque asume que cada versión ha guardado análisis por registro o permite reconstruirlo desde artefactos adicionales.

In [ ]:
# Placeholder razonable para ampliar con archivos `per_record_metrics.json` si se guardan en futuras ejecuciones.

## Conclusión comparativa

Criterios principales:
- macro-F1
- F1 por clases minoritarias
- estabilidad por registros
- coste de inferencia
- calidad de la explicación temporal

In [ ]:
if not comparison_df.empty:
    best_row = comparison_df.sort_values(["macro_f1", "weighted_f1"], ascending=False).iloc[0]
    print("Mejor versión por macro-F1:", best_row["version"])
    print(best_row)
else:
    print("Ejecuta primero v1, v2 y v3 para producir las métricas comparativas.")